## Отбор данных для датасета

In [ ]:
from pathlib import Path
import platform
import sys

OT_RELEASE = "26.09"
OT_BASE_URL = f"https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/{OT_RELEASE}/output/"
OT_DATASETS = (
    "drug_molecule",
    "target",
    "drug_mechanism_of_action"
)

ROOT_PATH = Path.cwd().resolve()

OR_DATA_DIR = ROOT_PATH / "data" / "open-targets" / OT_RELEASE
EDA_DATASET_DIR = ROOT_PATH / "data" / "eda" / OT_RELEASE

OR_DATA_DIR.mkdir(parents=True, exist_ok=True)
EDA_DATASET_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import re
import shutil
from urllib.parse import unquote, urljoin, urlparse
from urllib.request import urlopen


def download_dataset(dataset_name):
    dataset_url = urljoin(OT_BASE_URL, f"{dataset_name}/")

    with urlopen(dataset_url, timeout=120) as response:
        directory_html = response.read().decode("utf-8")

    parquet_links = sorted(
        set(re.findall(r'href="([^"]+\.parquet)"', directory_html))
    )
    if not parquet_links:
        raise FileNotFoundError(f"Parquet-файлы не найдены: {dataset_url}")

    dataset_dir = OR_DATA_DIR / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    local_paths = []

    for index, parquet_link in enumerate(parquet_links, start=1):
        file_url = urljoin(dataset_url, parquet_link)
        file_name = Path(unquote(urlparse(file_url).path)).name
        destination = dataset_dir / file_name

        if destination.exists():
            print(f"[{index}/{len(parquet_links)}] {dataset_name}: уже скачан {file_name}")
        else:
            print(f"[{index}/{len(parquet_links)}] {dataset_name}: скачиваем {file_name}")
            temporary = destination.with_suffix(destination.suffix + ".part")
            with urlopen(file_url, timeout=180) as response, temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
            temporary.replace(destination)

        local_paths.append(destination)

    return local_paths


source_files = {}

for dataset_name in OT_DATASETS:
    raw_data = download_dataset(dataset_name)
    source_files[dataset_name] = raw_data

print("\nЗагрузка завершена.")

for dataset_name, paths in source_files.items():
    total_size_mb = sum(path.stat().st_size for path in paths) / 1024**2
    print(f"{dataset_name}: {len(paths)} файлов, {total_size_mb:.1f} MiB")

[1/3] target: скачиваем part-00000-ddfc1f8c-60a1-40fe-885a-2c1dd3f9a9c1-c000.zstd.parquet
[2/3] target: скачиваем part-00001-ddfc1f8c-60a1-40fe-885a-2c1dd3f9a9c1-c000.zstd.parquet
[3/3] target: скачиваем part-00002-ddfc1f8c-60a1-40fe-885a-2c1dd3f9a9c1-c000.zstd.parquet
[1/1] drug_mechanism_of_action: скачиваем 00000000.parquet

Загрузка завершена.
drug_molecule: 1 файлов, 6.5 MiB
target: 3 файлов, 20.6 MiB
drug_mechanism_of_action: 1 файлов, 0.3 MiB


In [3]:
import duckdb


DUCKDB_CONNECT = duckdb.connect()

for dataset_name in OT_DATASETS:
    all_parquet_files = str(OR_DATA_DIR / dataset_name / "*.parquet")

    if not list((OR_DATA_DIR / dataset_name).glob("*.parquet")):
        raise FileNotFoundError(f"Нет файлов для датасета {dataset_name}. Сначала выполните загрузку.")
    
    query = (
        f"CREATE OR REPLACE VIEW {dataset_name} "
        f"AS SELECT * FROM read_parquet('{all_parquet_files}')"
    )
    DUCKDB_CONNECT.execute(query)

for dataset_name in OT_DATASETS:
    rows_count = DUCKDB_CONNECT.execute(
        f"SELECT count(*) FROM {dataset_name}"
    ).fetchall()[0][0]
    print(f"{dataset_name}: {rows_count:,} строк")

for dataset_name in ("drug_molecule", "target"):
    rows_count, unique_ids, nonnull_ids = DUCKDB_CONNECT.execute(
        f"SELECT count(*), count(DISTINCT id), count(id) FROM {dataset_name}"
    ).fetchone()
    assert rows_count == unique_ids == nonnull_ids, (
        f"В {dataset_name} есть пустые или повторяющиеся id"
    )

drug_molecule: 19,170 строк
target: 78,733 строк
drug_mechanism_of_action: 6,092 строк


In [4]:
import math
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors


DESCRIPTOR_COLUMNS = (
    "molecular_weight", "logp", "tpsa", "h_bond_donors",
    "h_bond_acceptors", "rotatable_bonds", "aromatic_rings",
)


def query_rows(sql, parameters=None):
    result = DUCKDB_CONNECT.execute(sql, parameters or [])
    columns = [column[0] for column in result.description]
    return [dict(zip(columns, row, strict=True)) for row in result.fetchall()]


def describe_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return {"structure_status": "missing_smiles", "has_dot": False}

    with rdBase.BlockLogs():
        try:
            molecule = Chem.MolFromSmiles(smiles)
        except (ValueError, RuntimeError):
            molecule = None

    if molecule is None or molecule.GetNumAtoms() == 0:
        return {"structure_status": "invalid_smiles", "has_dot": "." in smiles}

    values = (
        Descriptors.MolWt(molecule),
        Crippen.MolLogP(molecule),
        rdMolDescriptors.CalcTPSA(molecule),
        Lipinski.NumHDonors(molecule),
        Lipinski.NumHAcceptors(molecule),
        Lipinski.NumRotatableBonds(molecule),
        rdMolDescriptors.CalcNumAromaticRings(molecule),
    )
    if not all(math.isfinite(value) for value in values):
        return {"structure_status": "nonfinite_descriptor", "has_dot": "." in smiles}

    return {
        "structure_status": "valid",
        "has_dot": "." in smiles,
        "n_fragments": len(Chem.GetMolFrags(molecule)),
        "rdkitSmiles": Chem.MolToSmiles(molecule, isomericSmiles=True),
        **dict(zip(DESCRIPTOR_COLUMNS, values, strict=True)),
    }


def get_structure(identifier):
    if identifier not in structure_cache:
        structure_cache[identifier] = describe_smiles(molecules[identifier]["canonicalSmiles"])
    return structure_cache[identifier]


def resolve_parent(identifier, molecules):
    visited = set()
    while True:
        if identifier in visited:
            return None, "parent_cycle"
        visited.add(identifier)
        molecule = molecules.get(identifier)
        if molecule is None:
            return None, "parent_not_in_source"
        parent_id = molecule["parentId"]
        if not parent_id or parent_id == identifier:
            return identifier, "resolved"
        identifier = parent_id

molecule_rows = query_rows("""
    SELECT id, name, canonicalSmiles, drugType, parentId, maximumClinicalStage
    FROM drug_molecule ORDER BY id
""")
molecules = {row["id"]: row for row in molecule_rows}
small_molecules = [row for row in molecule_rows if row["drugType"] == "Small molecule"]
structure_cache = {}


mapping_records = []
issue_records = []
selected_compounds = {}
valid_original_smiles = 0
normalized_source_rows = 0

for molecule in small_molecules:
    original_id = molecule["id"]
    original = get_structure(original_id)
    status = original["structure_status"]
    resolved_id = None

    if status == "valid":
        valid_original_smiles += 1
        resolved_id, status = resolve_parent(original_id, molecules)

        if resolved_id is not None:
            candidate = molecules[resolved_id]
            normalized = get_structure(resolved_id)

            if candidate["drugType"] != "Small molecule":
                status = "parent_not_small_molecule"
            elif normalized["structure_status"] != "valid":
                status = "parent_" + normalized["structure_status"]
            else:
                normalized_source_rows += 1

                if normalized["n_fragments"] > 1:
                    status = "multicomponent_after_normalization"
                else:
                    status = "accepted"

                    selected_compounds[resolved_id] = {
                        "id": resolved_id,
                        "name": candidate["name"],
                        "canonicalSmiles": candidate["canonicalSmiles"],
                        "drugType": candidate["drugType"],
                        "parentId": candidate["parentId"],
                        "rdkitSmiles": normalized["rdkitSmiles"],
                        "maximumClinicalStage": candidate["maximumClinicalStage"],
                        "has_dot": normalized["has_dot"],
                        "n_fragments": normalized["n_fragments"],
                        **{column: normalized[column] for column in DESCRIPTOR_COLUMNS},
                    }

    mapping = {
        "original_id": original_id,
        "compound_id": resolved_id if status == "accepted" else None,
        "resolved_id": resolved_id,
        "parentId": molecule["parentId"],
        "canonicalSmiles": molecule["canonicalSmiles"],
        "original_status": original["structure_status"],
        "original_has_dot": original["has_dot"],
        "status": status,
    }
    mapping_records.append(mapping)

    reasons = []
    if original["structure_status"] != "valid":
        reasons.append(original["structure_status"])
    if original["has_dot"]:
        reasons.append("original_dot_smiles")
    if status != "accepted" and status not in reasons:
        reasons.append(status)
    issue_records.extend({**mapping, "reason": reason} for reason in reasons)

mapping_columns = [
    "original_id", "compound_id", "resolved_id", "parentId", "canonicalSmiles",
    "original_status", "original_has_dot", "status",
]
compound_columns = [
    "id", "name", "canonicalSmiles", "drugType", "parentId", "rdkitSmiles",
    "maximumClinicalStage", "has_dot", "n_fragments", *DESCRIPTOR_COLUMNS,
]

compound_mapping_df = pd.DataFrame(mapping_records, columns=mapping_columns)
compound_issues_df = pd.DataFrame(issue_records, columns=[*mapping_columns, "reason"])
compounds_df = pd.DataFrame(selected_compounds.values(), columns=compound_columns)

form_counts = (
    compound_mapping_df.loc[compound_mapping_df["status"] == "accepted"]
    .groupby("compound_id").size()
)
compounds_df["n_source_forms"] = compounds_df["id"].map(form_counts).fillna(0).astype("int64")
compounds_df = compounds_df.sort_values("id").reset_index(drop=True)
compound_mapping_df = compound_mapping_df.sort_values("original_id").reset_index(drop=True)
compound_issues_df = compound_issues_df.sort_values(["original_id", "reason"]).reset_index(drop=True)

for table_name, frame in {
    "compounds": compounds_df,
    "compound_mapping": compound_mapping_df,
    "compound_issues": compound_issues_df,
}.items():
    DUCKDB_CONNECT.register("temporary_frame", frame)
    DUCKDB_CONNECT.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM temporary_frame")
    DUCKDB_CONNECT.unregister("temporary_frame")

chemistry_stats = {
    "all_molecules": len(molecule_rows),
    "small_molecules": len(small_molecules),
    "valid_original_smiles": valid_original_smiles,
    "resolved_valid_small_parent": normalized_source_rows,
    "accepted_source_rows": int((compound_mapping_df["status"] == "accepted").sum()),
    "compounds": len(compounds_df),
}
display(pd.DataFrame([chemistry_stats]))
display(compound_mapping_df["status"].value_counts().rename_axis("status").reset_index(name="n"))

,all_molecules,small_molecules,valid_original_smiles,resolved_valid_small_parent,accepted_source_rows,compounds
0,19170,14889,14582,14421,14250,12595


,status,n
0,accepted,14250
1,missing_smiles,307
2,multicomponent_after_normalization,171
3,parent_not_in_source,158
4,parent_not_small_molecule,3


### Подготовка белков-ферментов

Оставляем protein_coding мишени, относящиеся к классу Enzyme уровня l1. Дополнительно рассчитываем подсемейства ферментов и количество GO-, pathway- и субклеточных аннотаций.

In [5]:
enzyme_levels = [
    row[0]
    for row in DUCKDB_CONNECT.execute("""
        SELECT DISTINCT c.level
        FROM target t, unnest(t.targetClass) u(c)
        WHERE c.label = 'Enzyme'
        ORDER BY c.level
    """).fetchall()
]
assert enzyme_levels == ["l1"], f"Неожиданные уровни Enzyme: {enzyme_levels}"

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE proteins AS
    WITH classes AS (
        SELECT t.id AS gene_id, c.id AS path_id, c.label, c.level
        FROM target t, unnest(t.targetClass) u(c)
    ),
    enzyme_paths AS (
        SELECT DISTINCT gene_id, path_id
        FROM classes
        WHERE label = 'Enzyme' AND level = 'l1'
    ),
    class_summary AS (
        SELECT
            c.gene_id,
            count(DISTINCT c.path_id) AS n_class_paths,
            count(DISTINCT e.path_id) AS n_enzyme_class_paths,
            coalesce(list_sort(list(DISTINCT c.label) FILTER (
                WHERE e.path_id IS NOT NULL AND c.level = 'l2'
            )), []) AS enzyme_subfamilies,
            coalesce(list_sort(list(DISTINCT c.label) FILTER (
                WHERE e.path_id IS NOT NULL AND c.level = 'l3'
            )), []) AS enzyme_l3
        FROM classes c
        LEFT JOIN enzyme_paths e
            ON c.gene_id = e.gene_id AND c.path_id = e.path_id
        GROUP BY c.gene_id
    )
    SELECT
        t.id,
        t.approvedSymbol,
        t.approvedName,
        t.biotype,
        t.targetClass,
        t.subcellularLocations,
        t.go,
        t.pathways,
        s.enzyme_subfamilies,
        s.enzyme_l3,
        s.n_class_paths,
        s.n_enzyme_class_paths,
        coalesce(list_unique(list_transform(t.subcellularLocations, lambda x: x.location)), 0)
            AS n_subcellular_locations,
        coalesce(list_unique(list_transform(
            list_filter(t.go, lambda x: x.aspect = 'F'), lambda x: x.id
        )), 0) AS n_go_functions,
        coalesce(list_unique(list_transform(
            list_filter(t.go, lambda x: x.aspect = 'P'), lambda x: x.id
        )), 0) AS n_go_processes,
        coalesce(list_unique(list_transform(
            list_filter(t.go, lambda x: x.aspect = 'C'), lambda x: x.id
        )), 0) AS n_go_components,
        coalesce(list_unique(list_transform(t.pathways, lambda x: x.pathwayId)), 0)
            AS n_pathways
    FROM target t
    JOIN class_summary s ON t.id = s.gene_id
    WHERE t.biotype = 'protein_coding'
      AND len(list_filter(
          t.targetClass,
          lambda x: x.label = 'Enzyme' AND x.level = 'l1'
      )) > 0
    ORDER BY t.id
""")

source_counts = {
    table_name: DUCKDB_CONNECT.execute(
        f"SELECT count(*) FROM {table_name}"
    ).fetchone()[0]
    for table_name in OT_DATASETS
}

protein_stats = query_rows("""
    SELECT
        count(*) AS all_targets,
        count(*) FILTER (WHERE biotype = 'protein_coding') AS protein_coding,
        count(*) FILTER (
            WHERE biotype = 'protein_coding' AND coalesce(len(targetClass), 0) > 0
        ) AS protein_coding_with_target_class
    FROM target
""")[0]
protein_stats["enzymes_l1"] = DUCKDB_CONNECT.execute(
    "SELECT count(*) FROM proteins"
).fetchone()[0]

display(pd.DataFrame([protein_stats]))

,all_targets,protein_coding,protein_coding_with_target_class,enzymes_l1
0,78733,20107,5865,2416


### Подготовка связей

Оставляем механизмы типа single protein с одной непустой мишенью. После разворачивания ChEMBL ID соединяем записи с выбранными веществами и ферментами. Исходные строки сохраняются в link_provenance, итоговые уникальные пары — в compound_protein.

In [6]:
single_protein_values = [
    row[0]
    for row in DUCKDB_CONNECT.execute("""
        SELECT DISTINCT targetType
        FROM drug_mechanism_of_action
        WHERE lower(trim(targetType)) = 'single protein'
    """).fetchall()
]
assert single_protein_values == ["single protein"], single_protein_values

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE moa AS
    SELECT row_number() OVER (ORDER BY to_json(m)) AS source_row_id, m.*
    FROM drug_mechanism_of_action m
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE moa_audit AS
    SELECT *,
        CASE
            WHEN targetType IS DISTINCT FROM 'single protein' THEN 'not_single_protein'
            WHEN coalesce(len(targets), 0) = 0 THEN 'missing_target_mapping'
            WHEN len(targets) <> 1 THEN 'ambiguous_target_mapping'
            WHEN targets[1] IS NULL THEN 'null_target_mapping'
            WHEN coalesce(len(chemblIds), 0) = 0 THEN 'missing_compound_mapping'
            ELSE 'eligible'
        END AS status
    FROM moa
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE expanded_pairs AS
    SELECT
        source_row_id,
        original_compound_id,
        target_id,
        actionType,
        mechanismOfAction,
        targetType,
        "references"
    FROM moa_audit,
    unnest(chemblIds) compounds(original_compound_id),
    unnest(targets) target_ids(target_id)
    WHERE status = 'eligible'
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE pair_audit AS
    SELECT
        e.*,
        m.compound_id,
        CASE
            WHEN m.original_id IS NULL THEN 'compound_not_small_molecule_or_missing'
            WHEN m.status <> 'accepted' THEN 'compound_' || m.status
            ELSE 'accepted'
        END AS compound_status,
        CASE
            WHEN p.id IS NULL THEN 'target_not_selected_enzyme'
            ELSE 'accepted'
        END AS protein_status
    FROM expanded_pairs e
    LEFT JOIN compound_mapping m ON e.original_compound_id = m.original_id
    LEFT JOIN proteins p ON e.target_id = p.id
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE link_provenance AS
    SELECT
        source_row_id,
        original_compound_id,
        compound_id,
        target_id,
        actionType,
        mechanismOfAction,
        targetType,
        "references"
    FROM pair_audit
    WHERE compound_status = 'accepted' AND protein_status = 'accepted'
    ORDER BY compound_id, target_id, source_row_id, original_compound_id
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE compound_protein AS
    SELECT
        compound_id,
        target_id,
        coalesce(list_sort(list(DISTINCT actionType) FILTER (
            WHERE actionType IS NOT NULL
        )), []) AS actionType,
        coalesce(list_sort(list(DISTINCT mechanismOfAction) FILTER (
            WHERE mechanismOfAction IS NOT NULL
        )), []) AS mechanismOfAction,
        min(targetType) AS targetType,
        list_sort(list(DISTINCT original_compound_id)) AS original_compound_ids,
        count(*) AS n_source_links
    FROM link_provenance
    GROUP BY compound_id, target_id
    ORDER BY compound_id, target_id
""")

DUCKDB_CONNECT.execute("""
    CREATE OR REPLACE TABLE duplicate_pairs AS
    SELECT
        compound_id,
        target_id,
        count(*) AS n_source_links,
        count(DISTINCT original_compound_id) AS n_original_compounds,
        list_sort(list(DISTINCT actionType) FILTER (
            WHERE actionType IS NOT NULL
        )) AS action_types
    FROM link_provenance
    GROUP BY compound_id, target_id
    HAVING count(*) > 1
    ORDER BY n_source_links DESC, compound_id, target_id
""")

display(DUCKDB_CONNECT.execute("""
    SELECT status, count(*) AS n
    FROM moa_audit
    GROUP BY status
    ORDER BY n DESC
""").df())

display(DUCKDB_CONNECT.execute("""
    SELECT compound_status, protein_status, count(*) AS n
    FROM pair_audit
    GROUP BY compound_status, protein_status
    ORDER BY n DESC
""").df())

,status,n
0,eligible,4164
1,not_single_protein,1506
2,missing_target_mapping,412
3,ambiguous_target_mapping,10


,compound_status,protein_status,n
0,accepted,target_not_selected_enzyme,2177
1,accepted,accepted,1467
2,compound_not_small_molecule_or_missing,target_not_selected_enzyme,1041
3,compound_not_small_molecule_or_missing,accepted,345
4,compound_missing_smiles,target_not_selected_enzyme,13
5,compound_missing_smiles,accepted,3
6,compound_multicomponent_after_normalization,accepted,3
7,compound_multicomponent_after_normalization,target_not_selected_enzyme,3


### Воронка отбора и проверки качества

Фиксируем число объектов после каждого этапа и проверяем уникальность ключей, отсутствие пропусков, ссылочную целостность и конечность химических дескрипторов.

In [7]:
def table_count(table_name, where=None):
    condition = f" WHERE {where}" if where else ""
    return DUCKDB_CONNECT.execute(
        f"SELECT count(*) FROM {table_name}{condition}"
    ).fetchall()[0][0]


protein_stats = query_rows("""
    SELECT
        count(*) AS all_targets,
        count(*) FILTER (WHERE biotype = 'protein_coding') AS protein_coding,
        count(*) FILTER (
            WHERE biotype = 'protein_coding' AND coalesce(len(targetClass), 0) > 0
        ) AS protein_coding_with_target_class
    FROM target
""")[0]
protein_stats["enzymes_l1"] = table_count("proteins")

funnel_records = [
    ("compounds", "all_molecules", chemistry_stats["all_molecules"], "records"),
    ("compounds", "small_molecules", chemistry_stats["small_molecules"], "records"),
    ("compounds", "valid_original_smiles", chemistry_stats["valid_original_smiles"], "records"),
    ("compounds", "resolved_valid_small_parent", chemistry_stats["resolved_valid_small_parent"], "records"),
    ("compounds", "after_mixture_policy", chemistry_stats["accepted_source_rows"], "records"),
    ("compounds", "unique_selected_compound_ids", chemistry_stats["compounds"], "compounds"),
    ("proteins", "all_targets", protein_stats["all_targets"], "targets"),
    ("proteins", "protein_coding", protein_stats["protein_coding"], "targets"),
    ("proteins", "protein_coding_with_target_class", protein_stats["protein_coding_with_target_class"], "targets"),
    ("proteins", "enzymes_l1", protein_stats["enzymes_l1"], "targets"),
    ("moa", "all_mechanisms", table_count("moa"), "records"),
    ("moa", "single_protein_type", table_count("moa", "targetType = 'single protein'"), "records"),
    ("moa", "eligible_mechanisms", table_count("moa_audit", "status = 'eligible'"), "records"),
    ("links", "expanded_eligible_mechanisms", table_count("expanded_pairs"), "expanded_links"),
    ("links", "selected_compounds", table_count("pair_audit", "compound_status = 'accepted'"), "expanded_links"),
    ("links", "selected_compounds_and_enzymes", table_count("link_provenance"), "expanded_links"),
    ("links", "unique_compound_protein_pairs", table_count("compound_protein"), "unique_pairs"),
]

funnel_df = pd.DataFrame(
    [
        {"branch": branch, "step": step, "stage": stage, "n": n, "unit": unit}
        for step, (branch, stage, n, unit) in enumerate(funnel_records, start=1)
    ]
)
DUCKDB_CONNECT.register("funnel_frame", funnel_df)
DUCKDB_CONNECT.execute("CREATE OR REPLACE TABLE funnel AS SELECT * FROM funnel_frame")
DUCKDB_CONNECT.unregister("funnel_frame")
display(funnel_df)

,branch,step,stage,n,unit
0,compounds,1,all_molecules,19170,records
1,compounds,2,small_molecules,14889,records
2,compounds,3,valid_original_smiles,14582,records
3,compounds,4,resolved_valid_small_parent,14421,records
4,compounds,5,after_mixture_policy,14250,records
5,compounds,6,unique_selected_compound_ids,12595,compounds
6,proteins,7,all_targets,78733,targets
7,proteins,8,protein_coding,20107,targets
8,proteins,9,protein_coding_with_target_class,5865,targets
9,proteins,10,enzymes_l1,2416,targets


In [8]:
checks = {}

for table_name in ("drug_molecule", "target", "compounds", "proteins"):
    checks[f"{table_name}_null_ids"] = table_count(table_name, "id IS NULL")
    checks[f"{table_name}_duplicate_ids"] = DUCKDB_CONNECT.execute(
        f"SELECT count(*) - count(DISTINCT id) FROM {table_name}"
    ).fetchone()[0]

checks["duplicate_final_pairs"] = DUCKDB_CONNECT.execute("""
    SELECT count(*) FROM (
        SELECT compound_id, target_id
        FROM compound_protein
        GROUP BY compound_id, target_id
        HAVING count(*) > 1
    )
""").fetchone()[0]
checks["null_pair_ids"] = table_count(
    "compound_protein", "compound_id IS NULL OR target_id IS NULL"
)
checks["orphan_compounds"] = DUCKDB_CONNECT.execute("""
    SELECT count(*)
    FROM compound_protein r
    ANTI JOIN compounds c ON r.compound_id = c.id
""").fetchone()[0]
checks["orphan_proteins"] = DUCKDB_CONNECT.execute("""
    SELECT count(*)
    FROM compound_protein r
    ANTI JOIN proteins p ON r.target_id = p.id
""").fetchone()[0]
checks["wrong_compound_types"] = table_count(
    "compounds", "drugType IS DISTINCT FROM 'Small molecule'"
)
checks["wrong_protein_biotypes"] = table_count(
    "proteins", "biotype IS DISTINCT FROM 'protein_coding'"
)
checks["wrong_link_target_types"] = table_count(
    "compound_protein", "targetType IS DISTINCT FROM 'single protein'"
)

for descriptor in DESCRIPTOR_COLUMNS:
    checks[f"{descriptor}_missing_or_nonfinite"] = table_count(
        "compounds", f"{descriptor} IS NULL OR NOT isfinite({descriptor})"
    )

failed_checks = {name: value for name, value in checks.items() if value != 0}
if failed_checks:
    raise ValueError(f"Проверки качества не пройдены: {failed_checks}")

actual_sizes = {
    "compounds": table_count("compounds"),
    "proteins": table_count("proteins"),
    "compound_protein": table_count("compound_protein"),
    "link_provenance": table_count("link_provenance"),
    "linked_compounds": DUCKDB_CONNECT.execute(
        "SELECT count(DISTINCT compound_id) FROM compound_protein"
    ).fetchone()[0],
    "linked_proteins": DUCKDB_CONNECT.execute(
        "SELECT count(DISTINCT target_id) FROM compound_protein"
    ).fetchone()[0],
}

expected_sizes = {
    "compounds": 12595,
    "proteins": 2416,
    "compound_protein": 1202,
    "link_provenance": 1467,
    "linked_compounds": 806,
    "linked_proteins": 290,
}
assert actual_sizes == expected_sizes, f"Получены неожиданные размеры: {actual_sizes}"

print("Все проверки пройдены.")
display(pd.DataFrame([actual_sizes]))

Все проверки пройдены.


,compounds,proteins,compound_protein,link_provenance,linked_compounds,linked_proteins
0,12595,2416,1202,1467,806,290


### Сохранение результата

Сохраняем только три основные таблицы в формате CSV. Списковые поля записываются как JSON-строки, чтобы их можно было восстановить в следующем ноутбуке.

In [9]:
EDA_DATASET_DIR.mkdir(parents=True, exist_ok=True)

old_output_files = (
    "checks.json",
    "summary.json",
    "funnel.csv",
    "funnel.parquet",
    "compounds.parquet",
    "proteins.parquet",
    "compound_protein.parquet",
    "compound_mapping.parquet",
    "compound_issues.parquet",
    "moa_audit.parquet",
    "pair_audit.parquet",
    "link_provenance.parquet",
    "duplicate_pairs.parquet",
    "compounds.csv",
    "proteins.csv",
    "compound_protein.csv",
)

for file_name in old_output_files:
    (EDA_DATASET_DIR / file_name).unlink(missing_ok=True)

export_queries = {
    "compounds": """
        SELECT *
        FROM compounds
        ORDER BY id
    """,
    "proteins": """
        SELECT
            id,
            approvedSymbol,
            approvedName,
            biotype,
            to_json(enzyme_subfamilies) AS enzyme_subfamilies,
            to_json(enzyme_l3) AS enzyme_l3,
            n_class_paths,
            n_enzyme_class_paths,
            n_subcellular_locations,
            n_go_functions,
            n_go_processes,
            n_go_components,
            n_pathways
        FROM proteins
        ORDER BY id
    """,
    "compound_protein": """
        SELECT
            compound_id,
            target_id,
            to_json(actionType) AS actionType,
            to_json(mechanismOfAction) AS mechanismOfAction,
            targetType,
            to_json(original_compound_ids) AS original_compound_ids,
            n_source_links
        FROM compound_protein
        ORDER BY compound_id, target_id
    """,
}

for table_name, query in export_queries.items():
    output_path = EDA_DATASET_DIR / f"{table_name}.csv"
    escaped_path = str(output_path).replace("'", "''")
    DUCKDB_CONNECT.execute(
        f"COPY ({query}) TO '{escaped_path}' (FORMAT CSV, HEADER)"
    ).fetchall()

saved_files = sorted(path.name for path in EDA_DATASET_DIR.iterdir() if path.is_file())
print(f"Датасет сохранён в {EDA_DATASET_DIR}")
print("\n".join(saved_files))

Датасет сохранён в /Users/miziulin_e/Projects/x/main/notebooks/data/eda/26.09
compound_protein.csv
compounds.csv
proteins.csv


### Результат

Подготовлены три CSV: compounds, proteins и compound_protein. Дальнейший анализ выполняется в 02-eda.ipynb без повторной обработки исходных Open Targets файлов.